[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/energy-system-modeling/blob/main/notebooks/p3_generation/10_dispatch_and_carbon.ipynb)

# Dispatch, then a price on carbon

The two plants of the book's sections 9.3 and 9.4 - the **Houston CCGT** and a **coal plant**, called *San Antonio coal* below - plus 150 MW of wind, serving 1,000 MW for one hour. In Colab, *File › Save a copy in Drive* first so your edits are kept.

**Before you run anything:** coal fuel is about 37% cheaper per MMBtu than gas at the plant gate. Which plant runs first? Write it down.


## Setup

One cell, the same in every notebook here: it installs what Colab does not have, fetches this repository so `data/` and `src/` are present, and moves into this notebook's own folder so the relative paths below resolve. The notebook's own imports follow in the same cell.


In [1]:
# --- setup: generated by tools/sync_setup_cells.py -- do not edit here, edit that
# The same cell in every notebook in this series. It installs what Colab does
# not have, fetches the repository so that data/ and src/ are present, and moves
# into this notebook's own folder so the relative paths below resolve.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sear-labs/energy-system-modeling"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1]
NOTEBOOK_DIR = "notebooks/p3_generation"

# Pinned, per Part 1 rule 3: an unpinned install will one day pull a major
# version with a changed API and either break or silently alter the answer.
PINS = ["highspy>=1.11,<2", "pypsa>=1.3,<2"]

if "google.colab" in sys.modules:
    if PINS:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    # An ABSOLUTE base, so re-running this cell is safe. Colab's "Run all" is
    # commonly run twice, and a relative check would look for the clone inside
    # the folder it had already moved into -- cloning a second copy nested one
    # level down, then working from the wrong one.
    BASE = Path("/content") if Path("/content").is_dir() else Path.home()
    REPO_DIR = BASE / REPO_NAME
    if not REPO_DIR.exists():
        cloned = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            capture_output=True, text=True)
        if cloned.returncode != 0:
            raise SystemExit(
                "Could not clone " + REPO_URL + "\n"
                + (cloned.stderr or "").strip() + "\n\n"
                "If that says 'not found', the repository is still private.\n"
                "A raw file URL fails the same way, so there is no way around\n"
                "it: it has to be public before a student can run this.")
    os.chdir(REPO_DIR / NOTEBOOK_DIR)
elif Path.cwd().name != Path(NOTEBOOK_DIR).name:
    raise SystemExit(
        "Run this notebook from its own folder (" + NOTEBOOK_DIR + "),\n"
        "so that ../../src and ../../data resolve.")

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
print("working directory:", Path.cwd().name)

# --- end generated setup; the notebook's own imports follow ---

import pypsa

working directory: p3_generation


## Part A - The merit order, by hand

A plant's marginal cost is **heat rate × fuel price at the plant gate + variable O&M**. Capital cost is not in it.


In [2]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)  # PyPSA's notices about its next version

MMBTU_PER_MWH = 3.412               # 1 MWh of electricity = 3.412 MMBtu of heat

# fuel price at the plant gate, $/MMBtu
gas_gate  = 2.65 / 1.033 + 0.50     # $/Mcf -> $/MMBtu, plus pipeline
coal_gate = 18.76 / 15.97 + 0.75    # $/ton -> $/MMBtu, plus rail
print(f'gas  ${gas_gate:.3f}/MMBtu    coal ${coal_gate:.3f}/MMBtu')


gas  $3.065/MMBtu    coal $1.925/MMBtu


Then the marginal cost, in $/MWh of electricity.


In [3]:
gas_hr,  gas_vom  = 6.370, 2.10     # MMBtu/MWh, $/MWh
coal_hr, coal_vom = 10.745, 5.06

gas_mc  = gas_hr  * gas_gate  + gas_vom
coal_mc = coal_hr * coal_gate + coal_vom
print(f'gas  ${gas_mc:.2f}/MWh    coal ${coal_mc:.2f}/MWh')


gas  $21.63/MWh    coal $25.74/MWh


> Coal's fuel was the cheaper one. Which number in the formula outweighed it? Was your prediction right?


## Part B - The same thing in PyPSA

The solver never sees the words *gas* or *coal*. Each fuel gets its own bus, measured in MWh of heat, and each plant is a **Link** that turns heat into electricity at `efficiency = 3.412 / heat rate`.


In [4]:
# Step 1 - buses: one for electricity, one per fuel
gas_kg, coal_kg = 53.06, 97.72      # kg CO2 per MMBtu of fuel (EIA)

n = pypsa.Network()
n.add('Carrier', 'gas',  co2_emissions=gas_kg * MMBTU_PER_MWH / 1000)   # t CO2 per MWh of heat
n.add('Carrier', 'coal', co2_emissions=coal_kg * MMBTU_PER_MWH / 1000)  # used in Part C
n.add('Bus', 'Houston')
n.add('Bus', 'gas fuel',  carrier='gas')
n.add('Bus', 'coal fuel', carrier='coal')


**Step 2 - supply:** fuel at its plant-gate price per MWh of heat, and wind.


In [5]:
n.add('Generator', 'gas supply',  bus='gas fuel',  carrier='gas',
      p_nom=10_000, marginal_cost=gas_gate * MMBTU_PER_MWH)
n.add('Generator', 'coal supply', bus='coal fuel', carrier='coal',
      p_nom=10_000, marginal_cost=coal_gate * MMBTU_PER_MWH)
n.add('Generator', 'wind', bus='Houston', p_nom=150, marginal_cost=0)


**Step 3 - the plants are Links:** heat in at `bus0`, electricity out at `bus1`.


In [6]:
gas_eff, coal_eff = MMBTU_PER_MWH / gas_hr, MMBTU_PER_MWH / coal_hr

n.add('Link', 'Houston CCGT', bus0='gas fuel', bus1='Houston',
      efficiency=gas_eff, p_nom=600 / gas_eff,        # p_nom is heat IN
      marginal_cost=gas_vom * gas_eff)                 # charged per MWh of heat IN
n.add('Link', 'San Antonio coal', bus0='coal fuel', bus1='Houston',
      efficiency=coal_eff, p_nom=600 / coal_eff,
      marginal_cost=coal_vom * coal_eff)


**Step 4 - one hour of demand, and solve.**


In [7]:
n.add('Load', 'demand', bus='Houston', p_set=1000)
n.optimize(solver_name='highs')


Index(['Houston'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.io: Writing time: 0.05s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 5 primals, 13 duals
Objective: 1.94e+04
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper were not assigned to the network.


('ok', 'optimal')

**Step 5 - who ran, and what the last MWh cost.**


In [8]:
out_b = -n.links_t.p1.iloc[0]                   # MW of electricity from each plant
price_b = n.buses_t.marginal_price['Houston'].iloc[0]
print(out_b.round(1).to_string())
print('wind', round(n.generators_t.p['wind'].iloc[0], 1))
print(f'price at Houston ${price_b:.2f}/MWh')


name
Houston CCGT        600.0
San Antonio coal    250.0
wind 150.0
price at Houston $25.74/MWh


> Compare the price to Part A. Which plant set it, and why that one?
>
> Now change one number. What wellhead gas price would put coal first? Work it out from Part A's formula **before** you edit `gas_gate` and re-run.


## Part C - A price on carbon, then a cap

A cold winter doubles the plant-gate gas price. Start there.


In [9]:
# emissions per MWh of electricity = heat rate x carbon content
gas_t  = gas_hr  * gas_kg / 1000     # t CO2 / MWh
coal_t = coal_hr * coal_kg / 1000
print(f'gas {gas_t:.3f} t/MWh    coal {coal_t:.3f} t/MWh')

gas_gate_winter = 2 * gas_gate


gas 0.338 t/MWh    coal 1.050 t/MWh


**The price form:** add tau × emissions to each fuel's cost, and re-solve.


In [10]:
tau = 0                              # $ per tonne CO2 - try 0, then 20

n.generators.loc['gas supply', 'marginal_cost'] = (
    gas_gate_winter * MMBTU_PER_MWH + tau * n.carriers.at['gas', 'co2_emissions'])
n.generators.loc['coal supply', 'marginal_cost'] = (
    coal_gate * MMBTU_PER_MWH + tau * n.carriers.at['coal', 'co2_emissions'])
n.optimize(solver_name='highs')
out_c, tau_c = -n.links_t.p1.iloc[0], tau
print(out_c.round(1).to_string())


Index(['Houston'], dtype='object', name='name')


Index(['0'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.io: Writing time: 0.04s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 5 primals, 13 duals
Objective: 2.57e+04
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper were not assigned to the network.


name
Houston CCGT        250.0
San Antonio coal    600.0


> At \$0 coal runs first. Does \$20 flip it? Write the one-line equation for the carbon price where the two plants tie, solve it by hand, then set `tau` just above it and run.


In [11]:
# cap form: no price, a limit on tonnes - and read the limit's shadow price
tau = 0
n.generators.loc['gas supply', 'marginal_cost'] = gas_gate_winter * MMBTU_PER_MWH
n.generators.loc['coal supply', 'marginal_cost'] = coal_gate * MMBTU_PER_MWH

cap_t = 500                            # tonnes of CO2 in the hour
n.add('GlobalConstraint', 'co2 cap', type='primary_energy',
      carrier_attribute='co2_emissions', sense='<=', constant=cap_t,
      overwrite=True)                    # safe to re-run this cell
n.optimize(solver_name='highs')
out_cap = -n.links_t.p1.iloc[0]
mu_cap = abs(n.global_constraints.at['co2 cap', 'mu'])
print(out_cap.round(1).to_string())
print(f'cap shadow price ${mu_cap:.2f}/t')


Index(['Houston'], dtype='object', name='name')


Index(['0'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.io: Writing time: 0.09s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 5 primals, 14 duals
Objective: 3.04e+04
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper were not assigned to the network.


name
Houston CCGT        551.3
San Antonio coal    298.7
cap shadow price $21.65/t


> The cap's shadow price is a number you have already met in this notebook. Which one, and why should they agree?

*Before class: a data center signs for 300 MW at the Houston bus. Which plant serves its last megawatt - and what would a \$20 carbon price do to its bill?*

### Sources
- Plant numbers: the book's sections 9.3 and 9.4 (Texas 2023 fuel prices; heat rates and VOM from NREL ATB 2024).
- CO2 content: EIA carbon dioxide emission coefficients (natural gas 53.06, lignite 97.72 kg/MMBtu).


---
### Does the package agree?

`esm.dispatch` solves the same three hours with `scipy`, written the way Part A thinks: one variable per plant in MW of electricity, priced at heat rate × fuel + VOM. No fuel buses, no Links, no efficiencies. So the two can agree only if every conversion in Part B is right - fuel per MWh of heat, `p_nom` in heat units, VOM charged per MWh of heat in. Each is a place to be wrong by a factor of 3.412 and still get a plausible dispatch.

A price is a dual, and a dual is determined only when enough plants sit strictly between empty and full - one for the energy balance, a second whenever the cap binds. `assert_prices_are_unique` checks that first, so an edit that lands the load exactly on a plant's limit stops with the reason instead of a mismatch.


In [12]:
from esm.dispatch import Thermal, assert_prices_are_unique, solve_hour
from esm.tolerance import AGREEMENT_RTOL, relative

# everything handed over is read from this notebook, not typed again
wind_mw = n.generators.at['wind', 'p_nom']
load_mw = n.loads.at['demand', 'p_set']
gas_mw = n.links.at['Houston CCGT', 'p_nom'] * gas_eff
coal_mw = n.links.at['San Antonio coal', 'p_nom'] * coal_eff
coal = Thermal('San Antonio coal', coal_mw, coal_hr, coal_gate, coal_vom, coal_kg)
summer = [Thermal('Houston CCGT', gas_mw, gas_hr, gas_gate, gas_vom, gas_kg), coal]
winter = [Thermal('Houston CCGT', gas_mw, gas_hr, gas_gate_winter, gas_vom, gas_kg), coal]

pkg_b = solve_hour(summer, load_mw, wind_mw)
pkg_c = solve_hour(winter, load_mw, wind_mw, co2_price=tau_c)
pkg_cap = solve_hour(winter, load_mw, wind_mw, co2_cap=cap_t)
for plants, r in ((summer, pkg_b), (winter, pkg_c), (winter, pkg_cap)):
    assert_prices_are_unique(plants, r, wind_mw)   # compare only what is determined

checks = [('Part B price $/MWh', price_b, pkg_b.price)]
checks += [(f'Part B {p} MW', out_b[p], pkg_b.output[p]) for p in out_b.index]
checks += [(f'Part C {p} MW', out_c[p], pkg_c.output[p]) for p in out_c.index]
checks += [(f'cap {p} MW', out_cap[p], pkg_cap.output[p]) for p in out_cap.index]
checks += [('cap shadow price $/t', mu_cap, pkg_cap.cap_price)]
for label, hand, pkg in checks:
    print(f'{label:26s} {hand:10.3f} {pkg:10.3f}   {relative(hand, pkg):.1e}')

worst = max(relative(a, b) for _, a, b in checks)
assert worst < AGREEMENT_RTOL, f'notebook and package disagree by {worst:.2e}'
print(f'\nnotebook and package agree to {worst:.1e}')


Part B price $/MWh             25.741     25.741   1.4e-16
Part B Houston CCGT MW        600.000    600.000   0.0e+00
Part B San Antonio coal MW    250.000    250.000   0.0e+00
Part C Houston CCGT MW        250.000    250.000   0.0e+00
Part C San Antonio coal MW    600.000    600.000   0.0e+00
cap Houston CCGT MW           551.259    551.259   2.1e-16
cap San Antonio coal MW       298.741    298.741   3.8e-16
cap shadow price $/t           21.645     21.645   9.8e-16

notebook and package agree to 9.8e-16


---
## Explore - for your reflection

Nothing above changes. Each slider cell rebuilds the Part B network with your numbers and solves it when you press **Run Interact**. The **edit-and-run** cells ask you to change a number or write model code yourself. Answer the numbered questions in your reflection.

### Now the streamlined version

Part B's network is wrapped into a function here, because a slider rebuilds and re-solves it every time it moves - and you have built every component of it by hand above. With the defaults it gives back Part B's dispatch and price.

### Slide it: dispatch


In [13]:
from ipywidgets import interact_manual, FloatSlider
import logging, warnings
for name in ('pypsa', 'linopy'):
    logging.getLogger(name).setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message='.*include_objective_constant')

def dispatch(gas_wellhead=2.65, coal_mine=18.76, co2_price=0.0,
             load_mw=1000.0, wind_mw=150.0, coal_min=0.0, co2_cap=None):
    """Part B's network with your numbers. coal_min: share of the coal unit
    that must stay on. co2_cap: tonnes, or None for no cap."""
    g_gate = gas_wellhead / 1.033 + 0.50
    c_gate = coal_mine / 15.97 + 0.75
    g_eff, c_eff = MMBTU_PER_MWH / gas_hr, MMBTU_PER_MWH / coal_hr
    k = pypsa.Network()
    k.add('Carrier', 'gas',  co2_emissions=53.06 * MMBTU_PER_MWH / 1000)
    k.add('Carrier', 'coal', co2_emissions=97.72 * MMBTU_PER_MWH / 1000)
    k.add('Bus', 'Houston')
    k.add('Bus', 'gas fuel',  carrier='gas')
    k.add('Bus', 'coal fuel', carrier='coal')
    k.add('Generator', 'gas supply', bus='gas fuel', carrier='gas', p_nom=10_000,
          marginal_cost=(g_gate * MMBTU_PER_MWH
                         + co2_price * k.carriers.at['gas', 'co2_emissions']))
    k.add('Generator', 'coal supply', bus='coal fuel', carrier='coal', p_nom=10_000,
          marginal_cost=(c_gate * MMBTU_PER_MWH
                         + co2_price * k.carriers.at['coal', 'co2_emissions']))
    k.add('Generator', 'wind', bus='Houston', p_nom=wind_mw, marginal_cost=0)
    k.add('Link', 'Houston CCGT', bus0='gas fuel', bus1='Houston', efficiency=g_eff,
          p_nom=600 / g_eff, marginal_cost=gas_vom * g_eff)
    k.add('Link', 'San Antonio coal', bus0='coal fuel', bus1='Houston',
          efficiency=c_eff, p_nom=600 / c_eff, marginal_cost=coal_vom * c_eff,
          p_min_pu=coal_min)
    k.add('Load', 'demand', bus='Houston', p_set=load_mw)
    if co2_cap is not None:
        k.add('GlobalConstraint', 'co2 cap', type='primary_energy',
              carrier_attribute='co2_emissions', sense='<=', constant=co2_cap)
    status, condition = k.optimize(solver_name='highs', log_to_console=False)
    if condition != 'optimal':
        print(f'no dispatch meets {load_mw:.0f} MW with these rules: the solver says "{condition}"')
        return
    gas = -k.links_t.p1['Houston CCGT'].iloc[0]
    coal = -k.links_t.p1['San Antonio coal'].iloc[0]
    wind = k.generators_t.p['wind'].iloc[0]
    tonnes = gas * gas_t + coal * coal_t
    print(f'wind {wind:5.0f} MW   gas {gas:5.0f} MW   coal {coal:5.0f} MW   '
          f'CO2 {tonnes:5.0f} t')
    print(f'price at Houston ${k.buses_t.marginal_price["Houston"].iloc[0]:.2f}/MWh')
    if co2_cap is not None:
        print(f'cap shadow price ${abs(k.global_constraints.at["co2 cap", "mu"]):.2f}/t')

interact_manual(dispatch,
    gas_wellhead=FloatSlider(min=1, max=8, step=0.05, value=2.65, description='gas $/Mcf'),
    coal_mine=FloatSlider(min=5, max=60, step=0.5, value=18.76, description='coal $/ton'),
    co2_price=FloatSlider(min=0, max=60, step=0.5, value=0, description='CO2 $/t'),
    load_mw=FloatSlider(min=200, max=1500, step=50, value=1000, description='load MW'),
    wind_mw=FloatSlider(min=0, max=600, step=25, value=150, description='wind MW'),
    coal_min=FloatSlider(min=0, max=1, step=0.1, value=0, description='coal must-run'),
    co2_cap=FloatSlider(min=300, max=1000, step=25, value=1000, description='CO2 cap t'));


interactive(children=(FloatSlider(value=2.65, description='gas $/Mcf', max=8.0, min=1.0, step=0.05), FloatSlid…

> **1.** Find the wellhead gas price at which coal moves ahead of gas. Does it match what Part A's formula predicts?
>
> **2.** Slide the load from 400 to 1,500 MW in steps of 100. At which loads does the price jump, and which plant sets it at each step? What happens near the top of the range, and what is the solver telling you?
>
> **3.** Put gas at about $5.80/Mcf so coal runs first, then find the CO2 price that flips them. Now set wind to 0, then 600 MW. Does the flip price move? Why or why not?
>
> **4.** With winter gas and no CO2 price, tighten the cap from 1,000 t in steps of 50. Does its shadow price change smoothly, jump, or stay put? Find the smallest cap this fleet can meet at 1,000 MW, then check it by hand with Part C's emission rates.

### Edit and run: a must-run rule

Change the number in the call, not the function.


In [14]:
# coal_min is the share of the coal unit that must stay on, whatever it costs.
# Try 0.0, then 0.5, then 1.0, at 700 MW.
dispatch(load_mw=700, coal_min=0.0)


wind   150 MW   gas   550 MW   coal    -0 MW   CO2   186 t
price at Houston $21.63/MWh


> **5.** What happens to gas, to the price and to the tonnes of CO2 as the rule tightens? Who asks for a rule like this in a real grid, and what are they protecting?

### Edit and run: add a plant yourself

Copy `dispatch` into the cell below, rename it `dispatch_with_peaker`, and add a **gas peaker** at Houston: heat rate 9.8 MMBtu/MWh, VOM $4.50/MWh, 200 MW. It is one more `k.add('Link', ...)` fed from `gas fuel`: follow the CCGT's pattern for `efficiency`, `p_nom` (heat in) and `marginal_cost`.


In [15]:
# def dispatch_with_peaker(...):
#     ...
#
# dispatch_with_peaker(load_mw=1300)


> **6.** Before you run it: at 1,300 MW, which plant sets the price, and at what $/MWh? Then run it. If your number and the solver's differ, which line of your `add` is the likely culprit?
